In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [62]:
df=pd.read_csv('IMDB Dataset.csv')
df.describe()

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [64]:
df.head(2)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive


In [65]:
print(df['review'].iloc[0])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

In [66]:
#Reviews contains abbrevations thus we will discard them using regrex and also the stopwords
import re

def clean_review(text):

    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)

    return text.strip()
df['review'] = df['review'].apply(clean_review)

In [67]:
texts=df['review']
labels=df['sentiment'].map({'positive':1,'negative':0})

In [68]:
tokenizer=Tokenizer(num_words=10000)
tokenizer.fit_on_texts(texts)

In [69]:
sequences=tokenizer.texts_to_sequences(texts)

In [70]:
print(sequences[0])

[28, 4, 1, 77, 2037, 46, 1051, 11, 100, 149, 41, 3056, 394, 20, 229, 29, 3173, 32, 25, 204, 14, 10, 6, 613, 47, 592, 17, 68, 1, 88, 148, 11, 3218, 68, 44, 3056, 13, 91, 5324, 2, 135, 4, 559, 61, 265, 8, 204, 37, 1, 647, 141, 1723, 68, 10, 6, 23, 3, 116, 16, 1, 7805, 2306, 40, 10, 116, 2569, 56, 5847, 17, 5439, 5, 1455, 371, 40, 559, 91, 6, 3783, 8, 1, 355, 356, 4, 1, 647, 7, 6, 432, 3056, 14, 11, 6, 1, 357, 5, 1, 6747, 2515, 1032, 7, 2685, 1400, 22, 518, 34, 4618, 2436, 4, 1, 1183, 115, 30, 1, 6927, 27, 2881, 2, 385, 36, 6, 23, 297, 22, 1, 4836, 2885, 518, 6, 340, 5, 107, 8059, 4992, 7688, 2424, 2, 52, 36, 324, 8983, 7180, 2, 8577, 25, 112, 223, 240, 9, 60, 132, 1, 280, 1319, 4, 1, 116, 6, 679, 5, 1, 192, 11, 7, 266, 115, 77, 274, 569, 21, 2981, 818, 182, 1290, 4123, 16, 2471, 1216, 818, 1420, 818, 865, 3056, 152, 21, 939, 184, 1, 88, 394, 9, 123, 210, 3218, 68, 14, 36, 1604, 7, 13, 2215, 9, 411, 21, 132, 9, 13, 1569, 16, 7, 18, 14, 9, 290, 52, 9, 1404, 3, 1256, 16, 3056, 2, 190, 5, 1,

In [71]:
X=pad_sequences(sequences,maxlen=200)
y=labels.values

In [72]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

In [73]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN,Dense,Embedding,Input

In [74]:
model=Sequential()
model.add(Input(shape=(200,)))
model.add(Embedding(input_dim=10000,output_dim=64,input_length=200))
model.add(SimpleRNN(64))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [75]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [76]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 200, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 648,321 (2.47 MB)

 Trainable params: 648,321 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

In [82]:
# train RNN
history=model.fit(
    X_train,
    y_train,
    batch_size=128,
    epochs=20,
    validation_split=0.2

)

Epoch 1/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 1.0000 - loss: 1.3687e-04 - val_accuracy: 0.8224 - val_loss: 1.0019
Epoch 2/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 1.0000 - loss: 1.1050e-04 - val_accuracy: 0.8229 - val_loss: 1.0196
Epoch 3/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 1.0000 - loss: 9.0424e-05 - val_accuracy: 0.8240 - val_loss: 1.0367
Epoch 4/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 1.0000 - loss: 7.4980e-05 - val_accuracy: 0.8219 - val_loss: 1.0523
Epoch 5/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 1.0000 - loss: 6.2622e-05 - val_accuracy: 0.8207 - val_loss: 1.0681
Epoch 6/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 1.0000 - loss: 5.2491e-05 - val_accuracy: 0.8232 - val_loss: 1.0823
Epoch 7/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 1.0000 - loss: 4.4555e-05 - val_accuracy: 0.8217 - val_loss: 1.0986
Epoch 8/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 1.00

In [79]:
model.evaluate(X_test,y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8310 - loss: 0.9303


[0.9303340911865234, 0.8309999704360962]

In [83]:
def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=200)

    pred = model.predict(pad)[0][0]

    if pred > 0.5:
        return "Positive"
    else:
        return "Negative"

In [87]:
print(predict_sentiment("Worst movie ever. I hate this film."))
print(predict_sentiment("This movie is fantastic. Loved it."))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
Negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Positive
